In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
#importing seaborn for statistical plots
import seaborn as sns
# Import Linear Regression machine learning library
from sklearn.linear_model import LinearRegression

import statsmodels.formula.api as smf

In [5]:
#recomm_df = pd.read_csv('https://drive.google.com/file/d/1ClBptsK3V5KgKXtK2GSRzFNAW7GnTPDW/view?usp=sharing')
recomm_df = pd.read_csv('ratings_Electronics.csv',names=["userId","productId", "ratings" , "timestamp"])


In [6]:
recomm_df.shape
recomm_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7824482 entries, 0 to 7824481
Data columns (total 4 columns):
userId       object
productId    object
ratings      float64
timestamp    int64
dtypes: float64(1), int64(1), object(2)
memory usage: 238.8+ MB


In [7]:
#drop the columns id, date, zipcode
recomm_df = recomm_df.drop(["timestamp"],axis=1)

In [8]:
recomm_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7824482 entries, 0 to 7824481
Data columns (total 3 columns):
userId       object
productId    object
ratings      float64
dtypes: float64(1), object(2)
memory usage: 179.1+ MB


In [9]:
# get the number of review by userId
users_by_count = recomm_df.groupby('userId')['productId'].count().reset_index(name='counts') 

In [10]:
#consider only users having  more than 50 count
top_users = users_by_count[ users_by_count['counts'] > 50]


In [11]:
from surprise.model_selection import train_test_split
top_users_reviews = recomm_df[recomm_df['userId'].isin(top_users['userId'].values)]
top_users_reviews.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 122171 entries, 118 to 7824444
Data columns (total 3 columns):
userId       122171 non-null object
productId    122171 non-null object
ratings      122171 non-null float64
dtypes: float64(1), object(2)
memory usage: 3.7+ MB


In [14]:
%matplotlib inline

import pandas
from sklearn.cross_validation import train_test_split
import numpy as np
import time
from sklearn.externals import joblib


In [15]:
from surprise import Reader, Dataset
reader = Reader()
data = Dataset.load_from_df(top_users_reviews[['userId', 'productId', 'ratings']], reader)
data.read_ratings

<bound method Dataset.read_ratings of <surprise.dataset.DatasetAutoFolds object at 0x1a18448358>>

In [49]:
### 3. split the data into  train_data, test_data
from surprise.model_selection import train_test_split

trainset, testset = train_test_split(data, test_size=0.30)

In [50]:
## 4. most popular recommeded model
wavg = top_users_reviews.groupby('productId')['ratings'].mean().reset_index(name='avg rating').sort_values('avg rating', ascending=False)
wavg.head(10)

,productId,avg rating
47154,B00LKG1MC8,5.0
18649,B001UE6HZ2,5.0
18704,B001UKJ8FC,5.0
18703,B001UK6UO4,5.0
37782,B0082N9DGY,5.0
37783,B0082N9E9U,5.0
18698,B001UJJNNA,5.0
18695,B001UIPV28,5.0
18694,B001UIHAOK,5.0
37789,B0082PZ0LO,5.0


In [39]:
## 5. collabrative filtering

In [40]:
 from surprise import SVD, accuracy
#from surprise.cross_validation import cross_validate
algo = SVD()
algo.fit(trainset)
#cross_validate(algo, tain_data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

In [28]:
predictions = algo.test(testset)

In [29]:
from surprise import accuracy
accuracy.rmse(predictions)

RMSE: 0.9885


0.9885220587104852

In [36]:
### 6. Evaluate the model

In [35]:
import surprise
from surprise import evaluate

surprise.evaluate(algo, data, measures=[u'rmse', u'mae'], with_dump=False)

/anaconda3/lib/python3.7/site-packages/surprise/evaluate.py:66: UserWarning: The evaluate() method is deprecated. Please use model_selection.cross_validate() instead.
  'model_selection.cross_validate() instead.', UserWarning)
/anaconda3/lib/python3.7/site-packages/surprise/dataset.py:193: UserWarning: Using data.split() or using load_from_folds() without using a CV iterator is now deprecated. 
  UserWarning)


Evaluating RMSE, MAE of algorithm SVD.

------------
Fold 1
RMSE: 0.9842
MAE:  0.7274
------------
Fold 2
RMSE: 0.9707
MAE:  0.7197
------------
Fold 3
RMSE: 0.9869
MAE:  0.7308
------------
Fold 4
RMSE: 0.9797
MAE:  0.7241
------------
Fold 5
RMSE: 0.9804
MAE:  0.7239
------------
------------
Mean RMSE: 0.9804
Mean MAE : 0.7252
------------
------------


CaseInsensitiveDefaultDict(list,
                           {'rmse': [0.9841573164210741,
                             0.9707489058539341,
                             0.9868625650162233,
                             0.9797167163744788,
                             0.980356169763153],
                            'mae': [0.7273523683775864,
                             0.7196529232841347,
                             0.7307856156346574,
                             0.7240610985089742,
                             0.7239238751613208]})

In [58]:
def get_Iu(uid):
    """ return the number of items rated by given user
    args: 
      uid: the id of the user
    returns: 
      the number of items rated by the user
    """
    try:
        return len(trainset.ur[trainset.to_inner_uid(uid)])
    except ValueError: # user was not part of the trainset
        return 0
    
def get_Ui(iid):
    """ return number of users that have rated given item
    args:
      iid: the raw id of the item
    returns:
      the number of users that have rated the item.
    """
    try: 
        return len(trainset.ir[trainset.to_inner_iid(iid)])
    except ValueError:
        return 0
    
df = pd.DataFrame(predictions, columns=['uid', 'iid', 'rui', 'est', 'details'])
df['Iu'] = df.uid.apply(get_Iu)
df['Ui'] = df.iid.apply(get_Ui)
df['err'] = abs(df.est - df.rui)
best_predictions = df.sort_values(by='err')[:10]
worst_predictions = df.sort_values(by='err')[-10:]

In [59]:
### 7. best predictions
best_predictions

,uid,iid,rui,est,details,Iu,Ui,err
8448,A31N0XY2UTB25C,B00212NO6W,5.0,5.0,{'was_impossible': False},159,12,0.0
32290,A3S3R88HA0HZG3,B000BQ7GW8,5.0,5.0,{'was_impossible': False},93,38,0.0
25025,A5KMMY627T3W,B0007QKMQY,5.0,5.0,{'was_impossible': False},89,10,0.0
18386,A3IRA0BHI9NE9U,B004G6002M,5.0,5.0,{'was_impossible': False},41,39,0.0
14676,A3RGJ1FXOB1ZLL,B007TG8QRW,5.0,5.0,{'was_impossible': False},62,2,0.0
32243,AK9BXHEXOOM6Z,B000AP05BO,5.0,5.0,{'was_impossible': False},42,11,0.0
18387,AK3GKIV8DEY8B,B000GYI76Y,5.0,5.0,{'was_impossible': False},78,8,0.0
18388,A1E1LEVQ9VQNK,B00D50UNRM,5.0,5.0,{'was_impossible': False},140,9,0.0
12587,A3KNGMX2RVQG91,B001ENW61I,5.0,5.0,{'was_impossible': False},42,14,0.0
7992,A3D0UM4ZD2CMAW,B007P4VOWC,5.0,5.0,{'was_impossible': False},79,23,0.0


In [60]:
### 8. Summarise your insights.
print("Though the given data is huge.. clence the  data to take the cream of the recommendation system.")
print("And Surprise library used for getting best pediction based on highest number of users rated high ( 5 rating).")
print("This can be achieved by collabrative filtering, it supports different prediction algo.")




Though the given data is huge.. clence the  data to take the cream of the recommendation system.
And Surprise library used for getting best pediction based on highest number of users rated high ( 5 rating).
This can be achieved by collabrative filtering, it supports different prediction algo.
